## Value Investing


In [31]:
# importing necessary libraries
import pandas as pd
import numpy as np
import math
import yfinance as yf
from parso.python.tree import Lambda
from scipy import stats

In [32]:
#loading nifty data set
tickers=pd.read_csv(r"dataset/ind_nifty50list.csv")
tickers.head()

,Company Name,Industry,Symbol,Series,ISIN Code
0,Adani Enterprises Ltd.,Metals & Mining,ADANIENT,EQ,INE423A01024
1,Adani Ports and Special Economic Zone Ltd.,Services,ADANIPORTS,EQ,INE742F01042
2,Apollo Hospitals Enterprise Ltd.,Healthcare,APOLLOHOSP,EQ,INE437A01024
3,Asian Paints Ltd.,Consumer Durables,ASIANPAINT,EQ,INE021A01026
4,Axis Bank Ltd.,Financial Services,AXISBANK,EQ,INE238A01034


In [33]:
tickers['Symbol']=tickers['Symbol']+'.NS'
tickers=tickers.drop(['Industry','Series','ISIN Code'],axis=1)

In [34]:
def fetch_values_of_stock(ticker_list):
    value_col=["Ticker","Price","PE ratio","PB ratio"]
    value_df=pd.DataFrame(columns=value_col)
    for ticker in ticker_list:
        stock=yf.Ticker(ticker)
        price=stock.history(period="1d")['Close'].iloc[-1]
        financials=stock.financials
        balancesheet=stock.balance_sheet
        cashflow=stock.cashflow
        pe_ration=stock.info.get("forwardPE",np.nan)
        pb_ratio=stock.info.get("priceToBook",np.nan)

        value_df.loc[len(value_df)]=[
                ticker,
                price,
                pe_ration,
                pb_ratio,
                ]
    return value_df


In [35]:
ticker_list=tickers['Symbol'].values.tolist()
ticker_list

['ADANIENT.NS',
 'ADANIPORTS.NS',
 'APOLLOHOSP.NS',
 'ASIANPAINT.NS',
 'AXISBANK.NS',
 'BSE.NS',
 'BAJAJ-AUTO.NS',
 'BAJFINANCE.NS',
 'BAJAJFINSV.NS',
 'BEL.NS',
 'BHARTIARTL.NS',
 'CIPLA.NS',
 'COALINDIA.NS',
 'DRREDDY.NS',
 'EICHERMOT.NS',
 'ETERNAL.NS',
 'GRASIM.NS',
 'HCLTECH.NS',
 'HDFCBANK.NS',
 'HDFCLIFE.NS',
 'HINDALCO.NS',
 'HINDUNILVR.NS',
 'ICICIBANK.NS',
 'ITC.NS',
 'INFY.NS',
 'INDIGO.NS',
 'JSWSTEEL.NS',
 'JIOFIN.NS',
 'KOTAKBANK.NS',
 'LT.NS',
 'M&M.NS',
 'MARUTI.NS',
 'MAXHEALTH.NS',
 'NTPC.NS',
 'NESTLEIND.NS',
 'ONGC.NS',
 'POWERGRID.NS',
 'RELIANCE.NS',
 'SBILIFE.NS',
 'SHRIRAMFIN.NS',
 'SBIN.NS',
 'SUNPHARMA.NS',
 'TCS.NS',
 'TATACONSUM.NS',
 'TMPV.NS',
 'TATASTEEL.NS',
 'TECHM.NS',
 'TITAN.NS',
 'TRENT.NS',
 'ULTRACEMCO.NS']

In [42]:
ad=yf.Ticker("ADANIENT.NS")
ad.info

{'address1': 'Adani Corporate House',
 'address2': 'Shantigram Near Vaishnodevi Circle S. G. Highway, Khodiyar',
 'city': 'Ahmedabad',
 'zip': '382421',
 'country': 'India',
 'phone': '91 79 2656 5555',
 'fax': '91 79 2555 5500',
 'website': 'https://www.adanienterprises.com',
 'industry': 'Thermal Coal',
 'industryKey': 'thermal-coal',
 'industryDisp': 'Thermal Coal',
 'sector': 'Energy',
 'sectorKey': 'energy',
 'sectorDisp': 'Energy',
 'longBusinessSummary': 'Adani Enterprises Limited, together with its subsidiaries, operates in the new energy ecosystem, data center, airports, roads, copper and PVC, and other sectors in India and internationally. It operates through the Integrated Resources Management (IRM), Mining Services, Commercial Mining, New Energy Ecosystem, Airport, Road, copper, and Others segments. The company engages in the integrated ecosystem for manufacturing green hydrogen, including solar cells, modules, and wind turbine generators; construction, operation, and maint

In [36]:
#we generate the dataframe
df=fetch_values_of_stock(ticker_list)
df

,Ticker,Price,PE ratio,PB ratio
0,ADANIENT.NS,2706.300049,48.106297,3.730605
1,ADANIPORTS.NS,1942.699951,24.407686,4.377985
2,APOLLOHOSP.NS,8549.500000,39.163360,11.590358
3,ASIANPAINT.NS,2509.300049,41.587860,11.064568
4,AXISBANK.NS,1310.699951,11.216265,1.764283
5,BSE.NS,3816.600098,41.077686,17.025396
6,BAJAJ-AUTO.NS,10979.000000,21.719921,8.270945
7,BAJFINANCE.NS,1064.849976,21.180693,5.379280
8,BAJAJFINSV.NS,1836.900024,20.097374,3.503048
9,BEL.NS,394.549988,35.318687,10.967643


In [37]:
#checkin for null values and dealing wih them
df.isnull().sum()

Ticker      0
Price       0
PE ratio    0
PB ratio    0
dtype: int64

# We are gonna calculate the percentile for pb ratio and pe ratio in terms of other 50 stocks

In [38]:
percentile_metrics={
    "PE ratio":"PE ratio_percentile",
    "PB ratio":"PB ratio_percentile",
}


for metric,percentile in percentile_metrics.items():
    df[percentile]=df[metric].apply(lambda x: stats.percentileofscore(df[metric],x)/100)

In [39]:
df

,Ticker,Price,PE ratio,PB ratio,PE ratio_percentile,PB ratio_percentile
0,ADANIENT.NS,2706.300049,48.106297,3.730605,0.86,0.44
1,ADANIPORTS.NS,1942.699951,24.407686,4.377985,0.60,0.48
2,APOLLOHOSP.NS,8549.500000,39.163360,11.590358,0.76,0.90
3,ASIANPAINT.NS,2509.300049,41.587860,11.064568,0.82,0.88
4,AXISBANK.NS,1310.699951,11.216265,1.764283,0.14,0.14
5,BSE.NS,3816.600098,41.077686,17.025396,0.80,0.92
6,BAJAJ-AUTO.NS,10979.000000,21.719921,8.270945,0.48,0.76
7,BAJFINANCE.NS,1064.849976,21.180693,5.379280,0.46,0.64
8,BAJAJFINSV.NS,1836.900024,20.097374,3.503048,0.40,0.40
9,BEL.NS,394.549988,35.318687,10.967643,0.72,0.86


In [46]:
# we calculate the mean and use that for finding the best stocks
from statistics import mean

df['Value Score']=df[[value for value in percentile_metrics.values()]].mean(axis=1)
df.head()

,Ticker,Price,PE ratio,PB ratio,PE ratio_percentile,PB ratio_percentile,Value Score
0,ADANIENT.NS,2706.300049,48.106297,3.730605,0.86,0.44,0.65
1,ADANIPORTS.NS,1942.699951,24.407686,4.377985,0.60,0.48,0.54
2,APOLLOHOSP.NS,8549.500000,39.163360,11.590358,0.76,0.90,0.83
3,ASIANPAINT.NS,2509.300049,41.587860,11.064568,0.82,0.88,0.85
4,AXISBANK.NS,1310.699951,11.216265,1.764283,0.14,0.14,0.14


In [49]:
#sorting my value by value score
df=df.sort_values(by="Value Score",ascending=False)
df.head()

,Ticker,Price,PE ratio,PB ratio,PE ratio_percentile,PB ratio_percentile,Value Score
34,NESTLEIND.NS,1297.800049,52.320374,43.283090,0.94,1.00,0.97
48,TRENT.NS,3119.199951,60.835040,21.401030,0.98,0.94,0.96
47,TITAN.NS,5005.000000,56.044310,24.325165,0.96,0.96,0.96
15,ETERNAL.NS,363.450012,118.440865,10.375985,1.00,0.84,0.92
38,SBILIFE.NS,1748.000000,50.673832,8.545295,0.92,0.80,0.86


we would later add more metrics into it for further details and precison

In [52]:
#adding the position and per unit buys for top 10 stocks
df=df.head(10)
portfolio_size=int(input("Enter the amount to invest : "))
position_size=portfolio_size/len(df.index)
df['Number of shares to buy']=df['Price'].apply(lambda price: math.floor(
    position_size/price
))

# reseting indexes

df.reset_index(drop=True)
df

,Ticker,Price,PE ratio,PB ratio,PE ratio_percentile,PB ratio_percentile,Value Score,Number of shares to buy
0,NESTLEIND.NS,1297.800049,52.320374,43.283090,0.94,1.00,0.97,7
1,TRENT.NS,3119.199951,60.835040,21.401030,0.98,0.94,0.96,3
2,TITAN.NS,5005.000000,56.044310,24.325165,0.96,0.96,0.96,1
3,ETERNAL.NS,363.450012,118.440865,10.375985,1.00,0.84,0.92,27
4,SBILIFE.NS,1748.000000,50.673832,8.545295,0.92,0.80,0.86,5
5,BSE.NS,3816.600098,41.077686,17.025396,0.80,0.92,0.86,2
6,ASIANPAINT.NS,2509.300049,41.587860,11.064568,0.82,0.88,0.85,3
7,APOLLOHOSP.NS,8549.500000,39.163360,11.590358,0.76,0.90,0.83,1
8,INDIGO.NS,5430.000000,25.096903,42.035347,0.62,0.98,0.80,1
9,BEL.NS,394.549988,35.318687,10.967643,0.72,0.86,0.79,25
